# 1. Creation of the df


## 1.1. Creation of Thermic_cool df

In [ ]:
import os
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Alignment
from openpyxl import Workbook
from scipy.optimize import curve_fit

# Constants
TEMP_THRESHOLD = 32.7
MIN_TEMP = 15.0
# PTS_TEMP_RANGE = (32.6, 33.4)

# Chemin du dossier principal contenant les sous-dossiers pour chaque animal
folder_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot'

# Chemin du fichier animals_info.xlsx
info_file_path = os.path.join(folder_path, 'animals_info.xlsx')

# Chargement des informations sur les animaux
info_df = pd.read_excel(info_file_path)

# Assurez-vous que le nom des colonnes correspond bien
animal_column = 'Animal'
age_column = 'Age'
sexe_column = 'Sexe'
weight_column = 'Weight'

# Initialisation d'une liste pour stocker les DataFrames de chaque fichier
dataframes = []

# Fonction pour calculer le Z-score
def calculate_z_score_by_rec(df):
    """
    Calculer le Z-score pour chaque combinaison de 'rec' et 'animal' sur l'ensemble des trials.
    """
    df['Z-score'] = None  # Initialiser avec None

    # Grouper par 'rec' et 'animal'
    grouped = df.groupby(['rec', 'animal'])

    for (rec, animal), group in grouped:
        rec_mask = (df['rec'] == rec) & (df['animal'] == animal)

        # Calculer la moyenne et l'écart-type des valeurs de fluorescence pour chaque rec
        mean_fluo = df.loc[rec_mask, 'fluorescence thermo'].mean()
        std_fluo = df.loc[rec_mask, 'fluorescence thermo'].std()

        # Calculer le Z-score pour chaque ligne dans le groupe
        df.loc[rec_mask, 'Z-score'] = (df.loc[rec_mask, 'fluorescence thermo'] - mean_fluo) / std_fluo

    return df

# Fonction pour détecter les périodes "TS" et "TB"
def detect_ts_tb_periods(df):
    df['period'] = None
    period_number = 1
    i = 0

    while i < len(df):
        if df.loc[i, 'temperature'] < TEMP_THRESHOLD:
            start_idx = i
            while i < len(df) and df.loc[i, 'temperature'] < TEMP_THRESHOLD:
                i += 1

            if i - start_idx >= 30:
                min_temp = df.loc[start_idx:i, 'temperature'].min()
                if min_temp < MIN_TEMP:
                    end_idx = start_idx
                    while end_idx < i - 1 and df.loc[end_idx + 1, 'temperature'] <= df.loc[end_idx, 'temperature']:
                        end_idx += 1

                    df.loc[start_idx:end_idx, 'period'] = f'TS_{period_number}'
                    tb_start_idx = max(0, start_idx - 100)
                    df.loc[tb_start_idx:start_idx - 1, 'period'] = f'TB_{period_number}'

                    period_number += 1
                    i = end_idx + 1
                    continue
        i += 1

    return df


# Fonction pour détecter les périodes "PTS"
def detect_pts_periods(df):
    # Remplacer les valeurs manquantes de 'period' par 'None'
    df['period'] = df['period'].fillna('None')
    i = 0
    period_number = 1

    while i < len(df):
        # Détecter la fin de la période TS
        if 'TS_' in df.loc[i, 'period']:
            start_ts_idx = i
            while i < len(df) and 'TS_' in df.loc[i, 'period']:
                i += 1
            end_ts_idx = i - 1

            # Calculer la fin de la période PTS pour que TS + PTS = 101 lignes
            start_pts_idx = end_ts_idx + 1
            ts_length = end_ts_idx - start_ts_idx + 1
            pts_length = 101 - ts_length
            end_pts_idx = start_pts_idx + pts_length - 1

            # S'assurer que la période PTS ne dépasse pas la taille du DataFrame
            if end_pts_idx < len(df):
                # Appliquer 'PTS' uniquement aux lignes qui ne sont pas déjà 'TS'
                df.loc[start_pts_idx:end_pts_idx, 'period'] = df.loc[start_pts_idx:end_pts_idx, 'period'].apply(
                    lambda x: f'PTS_{period_number}' if 'None' in x else x
                )
                period_number += 1
                i = end_pts_idx + 1
            else:
                # Si la fin dépasse les limites du DataFrame, ajustez et arrêtez
                break
        else:
            i += 1

    return df


# Fonction pour détecter les périodes "ITST"
def detect_itst_periods(df):
    df['period'] = df['period'].fillna('None')
    itst_number = 1

    first_tb_idx = df[df['period'].str.contains('TB_', na=False)].index.min()
    if first_tb_idx > 0:
        df.loc[:first_tb_idx - 1, 'period'] = f'ITST_{itst_number}'
        itst_number += 1

    ts_tb_periods = df[df['period'].str.contains('TS_|TB_', na=False)]

    for i in range(len(ts_tb_periods) - 1):
        current_period = ts_tb_periods.iloc[i]
        next_period = ts_tb_periods.iloc[i + 1]

        if 'PTS_' in current_period['period'] and 'TB_' in next_period['period']:
            start_idx = current_period.name + 1
            end_idx = next_period.name - 1
            if end_idx > start_idx:
                df.loc[start_idx:end_idx, 'period'] = f'ITST_{itst_number}'
                itst_number += 1

    last_pts_idx = df[df['period'].str.contains('PTS_', na=False)].index.max()
    if last_pts_idx < len(df) - 1:
        df.loc[last_pts_idx + 1:, 'period'] = 'ITST_9'

    return df

# Nouvelle fonction pour définir les trials
def define_trials(df):
    df['trial'] = None
    trial_number = 1
    i = 0

    while i < len(df):
        if trial_number == 1:
            itst1 = df[df['period'] == 'ITST_1']
            tb1 = df[df['period'] == 'TB_1']
            ts1 = df[df['period'] == 'TS_1']
            pts1 = df[df['period'] == 'PTS_1']
            itst2 = df[df['period'] == 'ITST_2']

            trial_df = pd.concat([itst1, tb1, ts1, pts1, itst2])
            df.loc[trial_df.index, 'trial'] = f'Trial_{trial_number}'
            trial_number += 1
            i = trial_df.index[-1] + 1
        else:
            tb_n = df[df['period'] == f'TB_{trial_number}']
            ts_n = df[df['period'] == f'TS_{trial_number}']
            pts_n = df[df['period'] == f'PTS_{trial_number}']
            itst_nplus1 = df[df['period'] == f'ITST_{trial_number + 1}']

            trial_df = pd.concat([tb_n, ts_n, pts_n, itst_nplus1])
            df.loc[trial_df.index, 'trial'] = f'Trial_{trial_number}'
            trial_number += 1
            i = trial_df.index[-1] + 1

            if trial_number > 8:
                break

    return df

# Fonction pour calculer le Z-score normalisé par la médiane de TB pour chaque trial
def calculate_z_score_norm_by_trial(df):
    """
    Calculer le Z-score normalisé par la moyenne du Z-score de la période TB
    pour chaque 'trial', 'rec', et 'animal'.
    """
    df['Z-score_norm'] = None  # Initialiser avec None

    # Grouper par 'trial', 'rec', et 'animal'
    grouped = df.groupby(['trial', 'rec', 'animal'])

    for (trial, rec, animal), group in grouped:
        trial_mask = (df['trial'] == trial) & (df['rec'] == rec) & (df['animal'] == animal)

        tb_mask = trial_mask & df['period'].str.contains('TB_', na=False)
        ts_pts_mask = trial_mask & df['period'].str.contains('TS_|PTS_', na=False)

        if not tb_mask.any():
            continue

        # Calculer la moyenne des Z-scores pour la période TB
        tb_mean = df.loc[tb_mask, 'Z-score'].mean()

        # Appliquer la normalisation Z-score pour les périodes TB, TS et PTS du trial courant
        df.loc[tb_mask | ts_pts_mask, 'Z-score_norm'] = df.loc[tb_mask | ts_pts_mask, 'Z-score'] - tb_mean

    return df

# Fonction pour calculer la colonne Stim_Time
def calculate_stim_time(df):
    df['Stim_Time'] = np.nan  # Initialiser la colonne Stim_Time avec des NaN

    for trial in df['trial'].unique():
        trial_df = df[df['trial'] == trial]

        # Gestion de TB_ période
        tb_period = trial_df[trial_df['period'].str.contains('TB_')]
        if not tb_period.empty:
            tb_start_idx = tb_period.index[0]
            tb_end_idx = tb_period.index[-1]

            tb_duration = tb_end_idx - tb_start_idx + 1
            tb_time_values = np.linspace(-100 * tb_duration, -100, tb_duration)

            df.loc[tb_start_idx:tb_end_idx, 'Stim_Time'] = tb_time_values

        # Gestion de TS_ et PTS_ périodes
        ts_period = trial_df[trial_df['period'].str.contains('TS_')]
        pts_period = trial_df[trial_df['period'].str.contains('PTS_')]

        if not ts_period.empty and not pts_period.empty:
            ts_start_idx = ts_period.index[0]
            pts_end_idx = pts_period.index[-1]

            df.loc[ts_start_idx, 'Stim_Time'] = 0  # TS_ commence à 0
            pts_duration = pts_end_idx - ts_start_idx
            pts_time_values = np.linspace(0, 100 * pts_duration, pts_duration + 1)

            df.loc[ts_start_idx:pts_end_idx, 'Stim_Time'] = pts_time_values

        # Les périodes ITST sont laissées en NaN
        itst_period = trial_df[trial_df['period'].str.contains('ITST_')]
        if not itst_period.empty:
            df.loc[itst_period.index, 'Stim_Time'] = np.nan

    return df

# Parcours de tous les sous-dossiers dans le dossier principal
for root, dirs, files in os.walk(folder_path):
    animal_name = os.path.basename(root)
    files.sort()

    for file in files:
        if file.startswith('~$'):
            continue

        if 'full_traces_all_masks_cool' in file:
            condition_part = file.split('_full_traces_all_masks_cool')[0]
            condition = condition_part.split('_')[0]
            recording_name = file.split('_full_traces_all_masks_cool')[0]

            file_path = os.path.join(root, file)
            df = pd.read_excel(file_path)

            df_selected = df.iloc[:, [0, 2, 4]].copy()
            df_selected.columns = ['frame', 'temperature', 'fluorescence thermo']
            df_selected['time'] = df_selected['frame'] * 100
            df_selected['temperature'] = df_selected['temperature'].interpolate(method='linear', limit_direction='both')
            df_selected['rec'] = recording_name
            df_selected['animal'] = animal_name
            df_selected['age'] = None
            df_selected['sexe'] = None
            df_selected['weight'] = None
            df_selected['condition'] = condition
            df_selected['period'] = None
            df_selected['trial'] = None

            if animal_name in info_df[animal_column].values:
                animal_info = info_df[info_df[animal_column] == animal_name]
                if not animal_info.empty:
                    animal_info = animal_info.iloc[0]
                    df_selected['age'] = animal_info[age_column]
                    df_selected['sexe'] = animal_info[sexe_column]
                    df_selected['weight'] = animal_info[weight_column]
                else:
                    print(f"Aucune information trouvée pour l'animal: {animal_name}")
            else:
                print(f"Animal {animal_name} non trouvé dans 'animals_info.xlsx'")

            df_selected = calculate_z_score_by_rec(df_selected)
            df_selected = detect_ts_tb_periods(df_selected)
            df_selected = detect_pts_periods(df_selected)
            df_selected = detect_itst_periods(df_selected)
            df_selected = define_trials(df_selected)
            df_selected = calculate_z_score_norm_by_trial(df_selected)
            df_selected['time_sec'] = df_selected['time'] / 1000
            df_selected = calculate_stim_time(df_selected)  # Calculer Stim_Time ici

            # Réorganiser les colonnes pour insérer Stim_Time après time
            df_selected = df_selected[['animal', 'rec', 'age', 'sexe', 'weight', 'trial', 'period', 'condition', 'frame',
                                       'time', 'time_sec', 'Stim_Time', 'temperature', 'fluorescence thermo', 'Z-score', 'Z-score_norm']]

            dataframes.append(df_selected)

# Combiner tous les DataFrames en un seul DataFrame final
final_df = pd.concat(dataframes, ignore_index=True)

# Chemin du fichier de sortie
output_file_path = os.path.join(folder_path, 'Df_thermic_Zscore_classic_cool.xlsx')

# Créer un nouveau classeur Excel et ajouter la feuille avec les données combinées
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    final_df.to_excel(writer, sheet_name='Master_sheet', index=False)

# Charger le fichier Excel pour ajuster les colonnes et ajouter le rapport
wb = load_workbook(output_file_path)
ws = wb.active

# Fonction pour appliquer l'alignement aux cellules
def center_cells(sheet):
    for row in sheet.iter_rows():
        for cell in row:
            cell.alignment = Alignment(horizontal='center', vertical='center')

# Ajuster la largeur des colonnes
for col in ws.columns:
    max_length = 0
    column = col[0].column_letter  # Colonne actuelle (A, B, C, etc.)
    for cell in col:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(cell.value)
        except:
            pass
    adjusted_width = (max_length + 2)
    ws.column_dimensions[column].width = adjusted_width

# Centrer les données dans la feuille active
center_cells(ws)

# Créer le rapport dans une nouvelle feuille
ws2 = wb.create_sheet(title="Report")

# Compter le nombre de lignes par enregistrement et par animal
report_df = final_df.groupby(['animal', 'rec']).size().reset_index(name='row_count')

# Ajouter le rapport à la nouvelle feuille
for r in dataframe_to_rows(report_df, index=False, header=True):
    ws2.append(r)

# Sauvegarder le fichier avec les modifications
wb.save(output_file_path)


#### Without ITST

In [ ]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_thermic_Zscore_classic_cool.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_thermic_Zscore_classic_cool_stim.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où 'Stim_Time' n'est pas NaN
df_filtered = df.dropna(subset=['Stim_Time'])

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
df_filtered.to_excel(output_file_path, index=False)

print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")


## 1.2. Creation of Thermic_hot df

In [ ]:
import os
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Alignment
from openpyxl import Workbook
from scipy.optimize import curve_fit



# Constants
TEMP_THRESHOLD = 33.12
MAX_TEMP = 46.0



# Chemin du dossier principal contenant les sous-dossiers pour chaque animal
folder_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot'



# Chemin du fichier animals_info.xlsx
info_file_path = os.path.join(folder_path, 'animals_info.xlsx')



# Chargement des informations sur les animaux
info_df = pd.read_excel(info_file_path)



# Assurez-vous que le nom des colonnes correspond bien
animal_column = 'Animal'
age_column = 'Age'
sexe_column = 'Sexe'
weight_column = 'Weight'



# Initialisation d'une liste pour stocker les DataFrames de chaque fichier
dataframes = []



# Fonction pour calculer le Z-score
def calculate_z_score_by_rec(df):
    """
    Calculer le Z-score pour chaque combinaison de 'rec' et 'animal' sur l'ensemble des trials.
    """
    df['Z-score'] = None  # Initialiser avec None

    # Grouper par 'rec' et 'animal'
    grouped = df.groupby(['rec', 'animal'])

    for (rec, animal), group in grouped:
        rec_mask = (df['rec'] == rec) & (df['animal'] == animal)

        # Calculer la moyenne et l'écart-type des valeurs de fluorescence pour chaque rec
        mean_fluo = df.loc[rec_mask, 'fluorescence thermo'].mean()
        std_fluo = df.loc[rec_mask, 'fluorescence thermo'].std()

        # Calculer le Z-score pour chaque ligne dans le groupe
        df.loc[rec_mask, 'Z-score'] = (df.loc[rec_mask, 'fluorescence thermo'] - mean_fluo) / std_fluo

    return df



# Fonction pour détecter les périodes "TS"
def detect_ts_periods(df):
    df['period'] = None
    current_period = 1
    start_index = None

    for i in range(len(df)):
        if df['temperature'].iloc[i] > TEMP_THRESHOLD:
            if start_index is None:
                start_index = i

            if df['period'].iloc[i] is None:
                if i > 0 and df['temperature'].iloc[i] < df['temperature'].iloc[i - 1]:
                    period_length = i - start_index
                    if period_length >= 20:
                        df.loc[start_index:i - 1, 'period'] = f'TS_{current_period}'
                        current_period += 1
                        if current_period > 8:
                            current_period = 1
                    start_index = None
        else:
            start_index = None

    if start_index is not None:
        period_length = len(df) - start_index
        if period_length >= 20:
            df.loc[start_index:, 'period'] = f'TS_{current_period}'

    return df



# Fonction pour détecter les périodes TB
def detect_tb_periods(df):
    current_period = 1  # Commence à TB_1
    for i in range(len(df)):
        # Vérifier si une période TS est détectée
        if df['period'].iloc[i] == f'TS_{current_period}':
            tb_start_index = i - 100  # La période TB commence 100 lignes avant

            # Vérifiez que l'index de départ est valide
            if tb_start_index >= 0:
                df.loc[tb_start_index:i - 1, 'period'] = f'TB_{current_period}'
            current_period += 1  # Passer à la période suivante
            if current_period > 8:  # Réinitialiser à TB_1 après TB_8
                current_period = 1

    return df



# Fonction pour détecter les périodes PTS
def detect_pts_periods(df):
    current_period = 1  # Commence à PTS_1
    for i in range(len(df)):
        # Vérifier si une période TS est détectée
        if df['period'].iloc[i] == f'TS_{current_period}':
            ts_length = 0

            # Compter le nombre de lignes de la période TS
            for j in range(i, len(df)):
                if df['period'].iloc[j] == f'TS_{current_period}':
                    ts_length += 1
                else:
                    break

            # Définir la durée de la période PTS
            pts_length = 101 - ts_length
            pts_start_index = i + ts_length  # La période PTS commence juste après la période TS

            # Attribuer la période PTS
            if pts_start_index + pts_length <= len(df):
                df.loc[pts_start_index:pts_start_index + pts_length - 1, 'period'] = f'PTS_{current_period}'
            
            current_period += 1  # Passer à la période suivante
            if current_period > 8:  # Réinitialiser à PTS_1 après PTS_8
                current_period = 1

    return df



# Fonction pour détecter les périodes ITST
def detect_itst_periods(df):
    df['period'] = df['period'].fillna('None')
    itst_number = 1

    # Vérifier s'il existe des frames avec la valeur 0 et réinitialiser les périodes à partir de ce point
    frame_zero_indices = df[df['frame'] == 0].index

    # Réinitialiser le compteur ITST pour chaque frame où la valeur est 0
    for frame_zero_idx in frame_zero_indices:
        itst_number = 1  # Réinitialiser à ITST_1
        df.loc[frame_zero_idx:, 'period'] = df.loc[frame_zero_idx:, 'period'].fillna(f'ITST_{itst_number}')
        
        # Rechercher le premier indice de TB après frame_zero_idx
        first_tb_idx = df[(df.index >= frame_zero_idx) & (df['period'].str.contains('TB_', na=False))].index.min()
        
        # Ajouter ITST avant la première occurrence de TB après frame_zero_idx
        if pd.notna(first_tb_idx) and first_tb_idx > frame_zero_idx:
            df.loc[frame_zero_idx:first_tb_idx - 1, 'period'] = f'ITST_{itst_number}'
            itst_number += 1

        # Récupérer toutes les périodes TS et TB à partir de frame_zero_idx
        ts_tb_periods = df[(df.index >= frame_zero_idx) & df['period'].str.contains('TS_|TB_', na=False)]

        for i in range(len(ts_tb_periods) - 1):
            current_period = ts_tb_periods.iloc[i]
            next_period = ts_tb_periods.iloc[i + 1]

            # Si une période PTS est suivie d'une période TB, ajouter une période ITST
            if 'PTS_' in current_period['period'] and 'TB_' in next_period['period']:
                start_idx = current_period.name + 1
                end_idx = next_period.name - 1
                if end_idx > start_idx:
                    df.loc[start_idx:end_idx, 'period'] = f'ITST_{itst_number}'
                    itst_number += 1

        # Ajouter une période ITST après la dernière période PTS
        last_pts_idx = df[(df.index >= frame_zero_idx) & df['period'].str.contains('PTS_', na=False)].index.max()
        if pd.notna(last_pts_idx) and last_pts_idx < len(df) - 1:
            df.loc[last_pts_idx + 1:, 'period'] = f'ITST_{itst_number}'

    return df



# Nouvelle fonction pour définir les trials
def define_trials(df):
    df['trial'] = None
    trial_number = 1
    i = 0

    while i < len(df):
        if trial_number == 1:
            itst1 = df[df['period'] == 'ITST_1']
            tb1 = df[df['period'] == 'TB_1']
            ts1 = df[df['period'] == 'TS_1']
            pts1 = df[df['period'] == 'PTS_1']
            itst2 = df[df['period'] == 'ITST_2']

            trial_df = pd.concat([itst1, tb1, ts1, pts1, itst2])
            df.loc[trial_df.index, 'trial'] = f'Trial_{trial_number}'
            trial_number += 1
            i = trial_df.index[-1] + 1
        else:
            tb_n = df[df['period'] == f'TB_{trial_number}']
            ts_n = df[df['period'] == f'TS_{trial_number}']
            pts_n = df[df['period'] == f'PTS_{trial_number}']
            itst_nplus1 = df[df['period'] == f'ITST_{trial_number + 1}']

            trial_df = pd.concat([tb_n, ts_n, pts_n, itst_nplus1])
            df.loc[trial_df.index, 'trial'] = f'Trial_{trial_number}'
            trial_number += 1
            i = trial_df.index[-1] + 1

            if trial_number > 8:
                break

    return df



# Fonction pour calculer le Z-score normalisé par la médiane de TB pour chaque trial
def calculate_z_score_norm_by_trial(df):
    """
    Calculer le Z-score normalisé par la moyenne du Z-score de la période TB
    pour chaque 'trial', 'rec', et 'animal'.
    """
    df['Z-score_norm'] = None  # Initialiser avec None

    # Grouper par 'trial', 'rec', et 'animal'
    grouped = df.groupby(['trial', 'rec', 'animal'])

    for (trial, rec, animal), group in grouped:
        trial_mask = (df['trial'] == trial) & (df['rec'] == rec) & (df['animal'] == animal)

        tb_mask = trial_mask & df['period'].str.contains('TB_', na=False)
        ts_pts_mask = trial_mask & df['period'].str.contains('TS_|PTS_', na=False)

        if not tb_mask.any():
            continue

        # Calculer la moyenne des Z-scores pour la période TB
        tb_mean = df.loc[tb_mask, 'Z-score'].mean()

        # Appliquer la normalisation Z-score pour les périodes TB, TS et PTS du trial courant
        df.loc[tb_mask | ts_pts_mask, 'Z-score_norm'] = df.loc[tb_mask | ts_pts_mask, 'Z-score'] - tb_mean

    return df



# Fonction pour calculer la colonne Stim_Time
def calculate_stim_time(df):
    df['Stim_Time'] = np.nan  # Initialiser la colonne Stim_Time avec des NaN

    for trial in df['trial'].unique():
        trial_df = df[df['trial'] == trial]

        # Gestion de TB_ période
        tb_period = trial_df[trial_df['period'].str.contains('TB_')]
        if not tb_period.empty:
            tb_start_idx = tb_period.index[0]
            tb_end_idx = tb_period.index[-1]

            tb_duration = tb_end_idx - tb_start_idx + 1
            tb_time_values = np.linspace(-100 * tb_duration, -100, tb_duration)

            df.loc[tb_start_idx:tb_end_idx, 'Stim_Time'] = tb_time_values

        # Gestion de TS_ et PTS_ périodes
        ts_period = trial_df[trial_df['period'].str.contains('TS_')]
        pts_period = trial_df[trial_df['period'].str.contains('PTS_')]

        if not ts_period.empty and not pts_period.empty:
            ts_start_idx = ts_period.index[0]
            pts_end_idx = pts_period.index[-1]

            df.loc[ts_start_idx, 'Stim_Time'] = 0  # TS_ commence à 0
            pts_duration = pts_end_idx - ts_start_idx
            pts_time_values = np.linspace(0, 100 * pts_duration, pts_duration + 1)

            df.loc[ts_start_idx:pts_end_idx, 'Stim_Time'] = pts_time_values

        # Les périodes ITST sont laissées en NaN
        itst_period = trial_df[trial_df['period'].str.contains('ITST_')]
        if not itst_period.empty:
            df.loc[itst_period.index, 'Stim_Time'] = np.nan

    return df



# Parcours de tous les sous-dossiers dans le dossier principal
for root, dirs, files in os.walk(folder_path):
    animal_name = os.path.basename(root)
    files.sort()

    for file in files:
        if file.startswith('~$'):
            continue

        if 'full_traces_all_masks_hot' in file:
            # condition_part = file.split('_full_traces_all_masks')[0]
            # condition = condition_part.split('_')[0]
            # recording_name = file.split('_full_traces_all_masks')[0]

            # Diviser le nom du fichier pour obtenir la partie avant '_full_traces_all_masks'
            recording_part = file.split('_full_traces_all_masks_hot')[0]
            # Diviser cette partie pour obtenir les éléments spécifiques
            parts = recording_part.split('_')
            # Extraire la condition et le nom de l'enregistrement
            condition = parts[1]  # 'hot'
            recording_name = '_'.join(parts[1:])  # 'hot_1'


            file_path = os.path.join(root, file)
            df = pd.read_excel(file_path)

            df_selected = df.iloc[:, [0, 2, 4]].copy()
            df_selected.columns = ['frame', 'temperature', 'fluorescence thermo']
            df_selected['time'] = df_selected['frame'] * 100
            df_selected['temperature'] = df_selected['temperature'].interpolate(method='linear', limit_direction='both')
            df_selected['rec'] = recording_name
            df_selected['animal'] = animal_name
            df_selected['age'] = None
            df_selected['sexe'] = None
            df_selected['weight'] = None
            df_selected['condition'] = condition
            df_selected['period'] = None
            df_selected['trial'] = None

            if animal_name in info_df[animal_column].values:
                animal_info = info_df[info_df[animal_column] == animal_name]
                if not animal_info.empty:
                    animal_info = animal_info.iloc[0]
                    df_selected['age'] = animal_info[age_column]
                    df_selected['sexe'] = animal_info[sexe_column]
                    df_selected['weight'] = animal_info[weight_column]
                else:
                    print(f"Aucune information trouvée pour l'animal: {animal_name}")
            else:
                print(f"Animal {animal_name} non trouvé dans 'animals_info.xlsx'")

            df_selected = calculate_z_score_by_rec(df_selected)
            df_selected = detect_ts_periods(df_selected)
            df_selected = detect_tb_periods (df_selected)
            df_selected = detect_pts_periods(df_selected)
            df_selected = detect_itst_periods(df_selected)
            df_selected = define_trials(df_selected)
            df_selected = calculate_z_score_norm_by_trial(df_selected)
            df_selected['time_sec'] = df_selected['time'] / 1000
            df_selected = calculate_stim_time(df_selected)  # Calculer Stim_Time ici

            # Réorganiser les colonnes pour insérer Stim_Time après time
            df_selected = df_selected[['animal', 'rec', 'age', 'sexe', 'weight', 'trial', 'period', 'condition', 'frame',
                                       'time', 'time_sec', 'Stim_Time', 'temperature', 'fluorescence thermo', 'Z-score', 'Z-score_norm']]

            dataframes.append(df_selected)



# Combiner tous les DataFrames en un seul DataFrame final
final_df = pd.concat(dataframes, ignore_index=True)



# Chemin du fichier de sortie
output_file_path = os.path.join(folder_path, 'Df_hot_Zscore_classic_hot.xlsx')



# Créer un nouveau classeur Excel et ajouter la feuille avec les données combinées
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    final_df.to_excel(writer, sheet_name='Master_sheet', index=False)



# Charger le fichier Excel pour ajuster les colonnes et ajouter le rapport
wb = load_workbook(output_file_path)
ws = wb.active



# Fonction pour appliquer l'alignement aux cellules
def center_cells(sheet):
    for row in sheet.iter_rows():
        for cell in row:
            cell.alignment = Alignment(horizontal='center', vertical='center')



# Ajuster la largeur des colonnes
for col in ws.columns:
    max_length = 0
    column = col[0].column_letter  # Colonne actuelle (A, B, C, etc.)
    for cell in col:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(cell.value)
        except:
            pass
    adjusted_width = (max_length + 2)
    ws.column_dimensions[column].width = adjusted_width



# Centrer les données dans la feuille active
center_cells(ws)



# Créer le rapport dans une nouvelle feuille
ws2 = wb.create_sheet(title="Report")



# Compter le nombre de lignes par enregistrement et par animal
report_df = final_df.groupby(['animal', 'rec']).size().reset_index(name='row_count')



# Ajouter le rapport à la nouvelle feuille
for r in dataframe_to_rows(report_df, index=False, header=True):
    ws2.append(r)



# Sauvegarder le fichier avec les modifications
wb.save(output_file_path)


#### Without ITST

In [ ]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_hot_Zscore_classic_hot.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_hot_Zscore_classic_hot_stim.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où 'Stim_Time' n'est pas NaN
df_filtered = df.dropna(subset=['Stim_Time'])

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
df_filtered.to_excel(output_file_path, index=False)

print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")


#

#

# 2. Df creation for each animal with Mean-SEM

## 2.1. Df for each animal with Mean-SEM - COOL

In [ ]:
import pandas as pd
import os

# Chemin du fichier Excel à modifier
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_thermic_Zscore_classic_cool.xlsx'

# Charger le fichier Excel dans un DataFrame
df = pd.read_excel(file_path)

# Obtenir la liste des valeurs uniques dans la colonne 'animal'
unique_animals = df['animal'].unique()

# Extraire le nom du fichier sans extension
file_name, _ = os.path.splitext(os.path.basename(file_path))

# Répertoire principal de sauvegarde
main_save_directory = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/'
# Créer un sous-répertoire pour les fichiers filtrés
filtered_save_directory = os.path.join(main_save_directory, 'Df_Thermic_by_animal_cool')
if not os.path.exists(filtered_save_directory):
    os.makedirs(filtered_save_directory)

# Traiter chaque animal individuellement
for animal in unique_animals:
    # Filtrer le DataFrame pour ne garder que les lignes où 'animal' est l'animal actuel
    df_filtered = df[df['animal'] == animal]

    # Créer une chaîne pour le nom de l'animal à garder
    animals_str = animal

    # Créer le chemin du fichier Excel modifié pour l'animal
    output_file_path = os.path.join(filtered_save_directory, f'{file_name}_{animals_str}.xlsx')

    # Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
    df_filtered.to_excel(output_file_path, index=False)

    print(f"Le DataFrame filtré pour l'animal '{animal}' a été sauvegardé avec succès dans : {output_file_path}")


## 2.2. Df for each animal with Mean-SEM - HOT

In [ ]:
import pandas as pd
import os

# Chemin du fichier Excel à modifier
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_hot_Zscore_classic_hot.xlsx'

# Charger le fichier Excel dans un DataFrame
df = pd.read_excel(file_path)

# Obtenir la liste des valeurs uniques dans la colonne 'animal'
unique_animals = df['animal'].unique()

# Extraire le nom du fichier sans extension
file_name, _ = os.path.splitext(os.path.basename(file_path))

# Répertoire principal de sauvegarde
main_save_directory = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/'
# Créer un sous-répertoire pour les fichiers filtrés
filtered_save_directory = os.path.join(main_save_directory, 'Df_Thermic_by_animal_hot')
if not os.path.exists(filtered_save_directory):
    os.makedirs(filtered_save_directory)

# Traiter chaque animal individuellement
for animal in unique_animals:
    # Filtrer le DataFrame pour ne garder que les lignes où 'animal' est l'animal actuel
    df_filtered = df[df['animal'] == animal]

    # Créer une chaîne pour le nom de l'animal à garder
    animals_str = animal

    # Créer le chemin du fichier Excel modifié pour l'animal
    output_file_path = os.path.join(filtered_save_directory, f'{file_name}_{animals_str}.xlsx')

    # Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
    df_filtered.to_excel(output_file_path, index=False)

    print(f"Le DataFrame filtré pour l'animal '{animal}' a été sauvegardé avec succès dans : {output_file_path}")


#

#

# 3. Creation des df "Trial_MeanSEM"

## 3.1. Df Cool_Trial_MeanSEM"

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

# Répertoires de sauvegarde
input_directory = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_Thermic_by_animal_cool/'
main_output_directory = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/'
output_directory = os.path.join(main_output_directory, 'Df_cool_Trial_MeanSEM')

# Créer le répertoire de sortie s'il n'existe pas
if not os.path.exists(output_directory):
    os.makedirs(output_directory)


# Fonction pour détecter les périodes TS, TB, et PTS
def detect_periods(df):
    df['period'] = None
    i = 0
    while i < len(df):
        if df.iloc[i]['Mean_temperature'] < TEMP_THRESHOLD:
            start_idx = i
            while i < len(df) and df.iloc[i]['Mean_temperature'] < TEMP_THRESHOLD:
                i += 1
            end_idx = i

            if end_idx - start_idx >= 30:
                min_temp_idx = df.iloc[start_idx:end_idx]['Mean_temperature'].idxmin()
                min_temp = df.iloc[min_temp_idx]['Mean_temperature']

                if min_temp < MIN_TEMP:
                    # Marquer la période TS de start_idx jusqu'à la température minimale
                    df.loc[start_idx:min_temp_idx, 'period'] = 'TS'

                    # Marquer la période TB avant TS
                    tb_start_idx = max(0, start_idx - 100)
                    df.loc[tb_start_idx:start_idx - 1, 'period'] = 'TB'

                    # Marquer toutes les lignes après TS comme PTS
                    df.loc[min_temp_idx + 1:, 'period'] = 'PTS'
                break  # Sortir après avoir marqué TS et PTS

        i += 1

    return df

# Définir les seuils de température
TEMP_THRESHOLD = 32.7
MIN_TEMP = 15.0

# Trouver tous les fichiers correspondant au modèle
file_pattern = os.path.join(input_directory, 'Df_thermic_Zscore_classic_cool_*.xlsx')
files = glob.glob(file_pattern)

print(f"Fichiers trouvés : {files}")

for file_path in files:
    # Charger le fichier Excel dans un DataFrame
    df = pd.read_excel(file_path)

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.strip()

    # Définir les colonnes nécessaires
    required_columns = ['trial', 'Stim_Time', 'Z-score_norm', 'temperature']
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        print(f"Colonnes manquantes : {missing_columns}")
    else:
        # Filtrer les colonnes utiles
        df_filtered = df[required_columns]

        # Définir les valeurs de Stim_Time à considérer (de -10000 à 9300 avec un écart de 100)
        stim_time_range = range(-10000, 10100, 100)

        # Grouper par trial et Stim_Time, puis calculer la moyenne et l'écart-type du Z-score_norm
        df_grouped = df_filtered[df_filtered['Stim_Time'].isin(stim_time_range)].groupby('Stim_Time').agg({
            'Z-score_norm': ['mean', 'sem'],
            'temperature': 'mean'
        }).reset_index()

        # Aplatir les colonnes groupées
        df_grouped.columns = ['Stim_Time', 'Mean_Zscore_norm', 'SEM_Zscore_norm', 'Mean_temperature']


        # Détecter les périodes TS, TB, et PTS dans le DataFrame groupé
        df_grouped = detect_periods(df_grouped)
 

        # Créer le chemin de sortie avec le suffixe '_trial_mean.xlsx'
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        output_file_path = os.path.join(output_directory, f'{base_name}_trial_mean_SEM.xlsx')

        # Sauvegarder le DataFrame final avec les périodes détectées
        df_grouped.to_excel(output_file_path, index=False)
        print(f"Le DataFrame final pour '{file_path}' a été sauvegardé avec succès dans : {output_file_path}")


## 3.2. Df Hot_Trial_MeanSEM"

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

# Répertoires de sauvegarde
input_directory = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_Thermic_by_animal_hot/'
main_output_directory = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/'
output_directory = os.path.join(main_output_directory, 'Df_hot_Trial_MeanSEM')

# Créer le répertoire de sortie s'il n'existe pas
if not os.path.exists(output_directory):
    os.makedirs(output_directory)


# Définir les seuils de température
TEMP_THRESHOLD = 33.12
MAX_TEMP = 46.0

# Fonction pour détecter les périodes TS, TB, et PTS
def detect_periods(df):
    df['period'] = None
    i = 0
    while i < len(df):
        if df.iloc[i]['Mean_temperature'] > TEMP_THRESHOLD:
            start_idx = i
            while i < len(df) and df.iloc[i]['Mean_temperature'] > TEMP_THRESHOLD:
                i += 1
            end_idx = i

            if end_idx - start_idx >= 20:
                max_temp_idx = df.iloc[start_idx:end_idx]['Mean_temperature'].idxmax()
                max_temp = df.iloc[max_temp_idx]['Mean_temperature']

                if max_temp > MAX_TEMP:
                    # Marquer la période TS de start_idx jusqu'à la température minimale
                    df.loc[start_idx:max_temp_idx, 'period'] = 'TS'

                    # Marquer la période TB avant TS
                    tb_start_idx = max(0, start_idx - 100)
                    df.loc[tb_start_idx:start_idx - 1, 'period'] = 'TB'

                    # Marquer toutes les lignes après TS comme PTS
                    df.loc[max_temp_idx + 1:, 'period'] = 'PTS'
                break  # Sortir après avoir marqué TS et PTS

        i += 1

    return df

# Trouver tous les fichiers correspondant au modèle
file_pattern = os.path.join(input_directory, 'Df_hot_Zscore_classic_hot_*.xlsx')
files = glob.glob(file_pattern)

print(f"Fichiers trouvés : {files}")

for file_path in files:
    # Charger le fichier Excel dans un DataFrame
    df = pd.read_excel(file_path)

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.strip()

    # Définir les colonnes nécessaires
    required_columns = ['trial', 'Stim_Time', 'Z-score_norm', 'temperature']
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        print(f"Colonnes manquantes : {missing_columns}")
    else:
        # Filtrer les colonnes utiles
        df_filtered = df[required_columns]

        # Définir les valeurs de Stim_Time à considérer (de -10000 à 9300 avec un écart de 100)
        stim_time_range = range(-10000, 10100, 100)

        # Grouper par trial et Stim_Time, puis calculer la moyenne et l'écart-type du Z-score_norm
        df_grouped = df_filtered[df_filtered['Stim_Time'].isin(stim_time_range)].groupby('Stim_Time').agg({
            'Z-score_norm': ['mean', 'sem'],
            'temperature': 'mean'
        }).reset_index()

        # Aplatir les colonnes groupées
        df_grouped.columns = ['Stim_Time', 'Mean_Zscore_norm', 'SEM_Zscore_norm', 'Mean_temperature']


        # Détecter les périodes TS, TB, et PTS dans le DataFrame groupé
        df_grouped = detect_periods(df_grouped)
 

        # Créer le chemin de sortie avec le suffixe '_trial_mean.xlsx'
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        output_file_path = os.path.join(output_directory, f'{base_name}_trial_mean_SEM.xlsx')

        # Sauvegarder le DataFrame final avec les périodes détectées
        df_grouped.to_excel(output_file_path, index=False)
        print(f"Le DataFrame final pour '{file_path}' a été sauvegardé avec succès dans : {output_file_path}")


#

#

# 4. DF construction for the AUC

### Directory for quantifications

In [ ]:
import os

# Définir le chemin de base et le répertoire à créer
base_dir = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot"
output_dir = os.path.join(base_dir, "Quantif_by_Periods")

# Vérifier si le répertoire existe, sinon le créer
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Le répertoire '{output_dir}' a été créé avec succès.")
else:
    print(f"Le répertoire '{output_dir}' existe déjà.")

# For ploting
# Définir le chemin de base et le répertoire à créer
base_dir = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot"
output_dir = os.path.join(base_dir, "Plot_Quantif_by_Periods")

# Vérifier si le répertoire existe, sinon le créer
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Le répertoire '{output_dir}' a été créé avec succès.")
else:
    print(f"Le répertoire '{output_dir}' existe déjà.")

## 4.1. For cool stim

In [ ]:
import pandas as pd
import numpy as np
from scipy.integrate import trapezoid

# Charger le dataset depuis le fichier Excel
file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Df_thermic_Zscore_classic_cool_stim.xlsx"
df = pd.read_excel(file_path)

# Assurez-vous que la colonne 'Z-score_norm' est de type numérique
df['Z-score_norm'] = pd.to_numeric(df['Z-score_norm'], errors='coerce')

# Extraire les essais uniques (en fonction de 'animal', 'rec', 'trial')
trials_unique = df[['animal', 'rec', 'trial', 'age', 'sexe', 'weight']].drop_duplicates().reset_index(drop=True)

# Créer une liste pour stocker les résultats
results = []

# Fonction pour calculer l'aire sous la courbe (AUC)
def calculate_auc(zscore_values, time_values):
    # Calculer l'aire au-dessus et en dessous de la baseline (0)
    auc_positive = trapezoid(zscore_values[zscore_values > 0], x=time_values[zscore_values > 0])
    auc_negative = trapezoid(zscore_values[zscore_values < 0], x=time_values[zscore_values < 0])
    total_auc = auc_positive + auc_negative
    return total_auc

# Itérer sur chaque essai unique
for _, trial_row in trials_unique.iterrows():
    # Filtrer le DataFrame pour obtenir les données de cet essai unique
    trial_data = df[(df['animal'] == trial_row['animal']) & 
                    (df['rec'] == trial_row['rec']) & 
                    (df['trial'] == trial_row['trial'])]
    
    # Grouper les données filtrées par période
    grouped = trial_data.groupby('period')
    
    # Itérer sur chaque période ('TS', 'TB', 'PTS')
    for period, group in grouped:
        # Extraire les valeurs de Z-score
        zscore_values = group['Z-score_norm'].values
        time_values = group['time_sec'].values  # Assuming you have a time variable
        
        # Calculer l'aire sous la courbe (AUC)
        total_auc = calculate_auc(zscore_values, time_values)
        
        # Compter le nombre de lignes dans chaque période
        num_lines = len(group)
        
        # Calculer la durée de la période en ms et en s
        duration_ms = num_lines * 100  # Durée en ms
        duration_s = duration_ms / 1000  # Durée en s
        
        # Normaliser l'aire par la durée de la période (en secondes)
        auc_per_duration = total_auc / duration_s if duration_s > 0 else 0
        
        # Multiplication de la valeur de l'AUC pour visualisation
        auc10_per_duration = auc_per_duration * 10

        # Ajouter les résultats à la liste
        results.append({
            'Animal': trial_row['animal'],
            'Rec': trial_row['rec'],
            'Trial': trial_row['trial'],
            'Period': period,
            'AUC_Znorm_persec': auc10_per_duration,
            'Period_duration': duration_ms,  # Durée de la période en ms
            'Normalization_value': duration_s,  # Valeur utilisée pour normaliser en secondes
            'Age': trial_row['age'],
            'Sexe': trial_row['sexe'],
            'Weight': trial_row['weight']
        })

# Convertir la liste de résultats en DataFrame
df_auc_values = pd.DataFrame(results)

# Définir le chemin où sauvegarder le fichier Excel
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\AUC_persec_Znorm_by_Period_cool.xlsx"

# Sauvegarder le DataFrame dans un fichier Excel
df_auc_values.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les données ont été sauvegardées dans {output_file_path}")


### Supression des Periods index for stat on periods

In [ ]:
import pandas as pd

# Charger le fichier Excel
file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\AUC_persec_Znorm_by_Period_cool.xlsx"
df = pd.read_excel(file_path)

# Vérifier les colonnes disponibles
print(df.columns)

# Supposons que la colonne contenant les périodes s'appelle 'Period'
# Extraire les valeurs de 'Period' sans les suffixes '_index'
df['Period'] = df['Period'].str.extract(r'([A-Z]+)')[0]

# # Créer une nouvelle colonne 'Period_index' avec les indices
# df['Period_index'] = df['Period'].str.extract(r'_(\d+)')[0]

# Supprimer l'indice de la colonne 'Period'
# En utilisant str.replace pour supprimer l'indice dans 'Period'
df['Period'] = df['Period'].str.replace(r'_(\d+)', '', regex=True)

# Renommer le fichier de sortie
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\AUC_persec_Znorm_by_Period_wo_index_cool.xlsx"
df.to_excel(output_file_path, index=False)

print(f"Le fichier a été enregistré sous : {output_file_path}")


### Moyennage par animaux

In [ ]:
import pandas as pd

# Charger le DataFrame contenant les valeurs d'AUC
input_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\AUC_persec_Znorm_by_Period_wo_index_cool.xlsx"
df_auc_values = pd.read_excel(input_file_path)

# Calculer la moyenne des AUC par période, animal, sexe et âge
mean_auc_by_animal = df_auc_values.groupby(['Animal', 'Sexe', 'Age', 'Period'], as_index=False)['AUC_Znorm_persec'].mean()

# Définir le chemin où sauvegarder le fichier Excel des moyennes
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\AUC_persec_Znorm_by_Period_animal_cool.xlsx"

# Sauvegarder le DataFrame des moyennes dans un fichier Excel
mean_auc_by_animal.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les moyennes d'AUC par période, sexe et âge ont été sauvegardées dans {output_file_path}")


### Récupération des Data par animal pour period TS

In [ ]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\AUC_persec_Znorm_by_Period_animal_cool.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Quantif_by_Periods/AUC_persec_Znorm_TS_only_animal_cool.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où la colonne 'period' contient 'TS' (partout dans la cellule)
df_filtré = df[df['Period'].str.startswith("TS", na=False)]

# Vérifier le nombre de lignes filtrées
print(f"Nombre de lignes filtrées : {len(df_filtré)}")

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
if not df_filtré.empty:
    df_filtré.to_excel(output_file_path, index=False)
    print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")
else:
    print("Aucune ligne correspondante pour 'TS' trouvée.")

## 4.2. For hot stim

In [ ]:
import pandas as pd
import numpy as np
from scipy.integrate import trapezoid

# Charger le dataset depuis le fichier Excel
file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Df_hot_Zscore_classic_hot_stim.xlsx"
df = pd.read_excel(file_path)

# Assurez-vous que la colonne 'Z-score_norm' est de type numérique
df['Z-score_norm'] = pd.to_numeric(df['Z-score_norm'], errors='coerce')

# Extraire les essais uniques (en fonction de 'animal', 'rec', 'trial')
trials_unique = df[['animal', 'rec', 'trial', 'age', 'sexe', 'weight']].drop_duplicates().reset_index(drop=True)

# Créer une liste pour stocker les résultats
results = []

# Fonction pour calculer l'aire sous la courbe (AUC)
def calculate_auc(zscore_values, time_values):
    # Calculer l'aire au-dessus et en dessous de la baseline (0)
    auc_positive = trapezoid(zscore_values[zscore_values > 0], x=time_values[zscore_values > 0])
    auc_negative = trapezoid(zscore_values[zscore_values < 0], x=time_values[zscore_values < 0])
    total_auc = auc_positive + auc_negative
    return total_auc

# Itérer sur chaque essai unique
for _, trial_row in trials_unique.iterrows():
    # Filtrer le DataFrame pour obtenir les données de cet essai unique
    trial_data = df[(df['animal'] == trial_row['animal']) & 
                    (df['rec'] == trial_row['rec']) & 
                    (df['trial'] == trial_row['trial'])]
    
    # Grouper les données filtrées par période
    grouped = trial_data.groupby('period')
    
    # Itérer sur chaque période ('TS', 'TB', 'PTS')
    for period, group in grouped:
        # Extraire les valeurs de Z-score
        zscore_values = group['Z-score_norm'].values
        time_values = group['time_sec'].values  # Assuming you have a time variable
        
        # Calculer l'aire sous la courbe (AUC)
        total_auc = calculate_auc(zscore_values, time_values)
        
        # Compter le nombre de lignes dans chaque période
        num_lines = len(group)
        
        # Calculer la durée de la période en ms et en s
        duration_ms = num_lines * 100  # Durée en ms
        duration_s = duration_ms / 1000  # Durée en s
        
        # Normaliser l'aire par la durée de la période (en secondes)
        auc_per_duration = total_auc / duration_s if duration_s > 0 else 0
        
        # Multiplication de la valeur de l'AUC pour visualisation
        auc10_per_duration = auc_per_duration * 10

        # Ajouter les résultats à la liste
        results.append({
            'Animal': trial_row['animal'],
            'Rec': trial_row['rec'],
            'Trial': trial_row['trial'],
            'Period': period,
            'AUC_Znorm_persec': auc10_per_duration,
            'Period_duration': duration_ms,  # Durée de la période en ms
            'Normalization_value': duration_s,  # Valeur utilisée pour normaliser en secondes
            'Age': trial_row['age'],
            'Sexe': trial_row['sexe'],
            'Weight': trial_row['weight']
        })

# Convertir la liste de résultats en DataFrame
df_auc_values = pd.DataFrame(results)

# Définir le chemin où sauvegarder le fichier Excel
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\AUC_persec_Znorm_by_Period_hot.xlsx"

# Sauvegarder le DataFrame dans un fichier Excel
df_auc_values.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les données ont été sauvegardées dans {output_file_path}")


### Supression des Periods index for stat on periods

In [ ]:
import pandas as pd

# Charger le fichier Excel
file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\AUC_persec_Znorm_by_Period_hot.xlsx"
df = pd.read_excel(file_path)

# Vérifier les colonnes disponibles
print(df.columns)

# Supposons que la colonne contenant les périodes s'appelle 'Period'
# Extraire les valeurs de 'Period' sans les suffixes '_index'
df['Period'] = df['Period'].str.extract(r'([A-Z]+)')[0]

# # Créer une nouvelle colonne 'Period_index' avec les indices
# df['Period_index'] = df['Period'].str.extract(r'_(\d+)')[0]

# Supprimer l'indice de la colonne 'Period'
# En utilisant str.replace pour supprimer l'indice dans 'Period'
df['Period'] = df['Period'].str.replace(r'_(\d+)', '', regex=True)

# Renommer le fichier de sortie
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\AUC_persec_Znorm_by_Period_wo_index_hot.xlsx"
df.to_excel(output_file_path, index=False)

print(f"Le fichier a été enregistré sous : {output_file_path}")


### Moyennage par animaux

In [ ]:
import pandas as pd

# Charger le DataFrame contenant les valeurs d'AUC
input_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\AUC_persec_Znorm_by_Period_wo_index_hot.xlsx"
df_auc_values = pd.read_excel(input_file_path)

# Calculer la moyenne des AUC par période, animal, sexe et âge
mean_auc_by_animal = df_auc_values.groupby(['Animal', 'Sexe', 'Age', 'Period'], as_index=False)['AUC_Znorm_persec'].mean()

# Définir le chemin où sauvegarder le fichier Excel des moyennes
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\AUC_persec_Znorm_by_Period_animal_hot.xlsx"

# Sauvegarder le DataFrame des moyennes dans un fichier Excel
mean_auc_by_animal.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les moyennes d'AUC par période, sexe et âge ont été sauvegardées dans {output_file_path}")


### Récupération des Data par animal pour period TS

In [ ]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\AUC_persec_Znorm_by_Period_animal_hot.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Quantif_by_Periods/AUC_persec_Znorm_TS_only_animal_hot.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où la colonne 'period' contient 'TS' (partout dans la cellule)
df_filtré = df[df['Period'].str.startswith("TS", na=False)]

# Vérifier le nombre de lignes filtrées
print(f"Nombre de lignes filtrées : {len(df_filtré)}")

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
if not df_filtré.empty:
    df_filtré.to_excel(output_file_path, index=False)
    print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")
else:
    print("Aucune ligne correspondante pour 'TS' trouvée.")

#

#

# 5. DF construction for the Max

## 5.1. For cool stim

In [ ]:
import pandas as pd
import numpy as np

# Charger le dataset depuis le fichier Excel
file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Df_thermic_Zscore_classic_cool_stim.xlsx"
df = pd.read_excel(file_path)

# Assurez-vous que la colonne 'Z-score_norm' est de type numérique
df['Z-score_norm'] = pd.to_numeric(df['Z-score_norm'], errors='coerce')

# Extraire les trials uniques (en fonction de 'animal', 'Trial', 'trial')
# On inclut aussi les colonnes 'age', 'sexe', et 'weight' dans les données uniques
trials_unique = df[['animal', 'rec', 'trial', 'age', 'sexe', 'weight']].drop_duplicates().reset_index(drop=True)

# Créer une liste pour stocker les résultats
results = []

# Itérer sur chaque trial unique
for _, trial_row in trials_unique.iterrows():
    # Filtrer le DataFrame pour obtenir les données de ce trial unique
    trial_data = df[(df['animal'] == trial_row['animal']) & 
                    (df['rec'] == trial_row['rec']) & 
                    (df['trial'] == trial_row['trial'])]
    
    # Grouper les données filtrées par période
    grouped = trial_data.groupby('period')
    
    # Itérer sur chaque période ('TS', 'TB', 'PTS')
    for period, group in grouped:
        # Trouver la ligne avec la valeur maximale de Z-score_norm
        max_row = group.loc[group['Z-score_norm'].idxmax()]
        
        # Extraire les informations nécessaires
        Max_Znorm = max_row['Z-score_norm']
        associated_temp = max_row['temperature']
        stim_time = max_row['Stim_Time']
        
        # Ajouter les résultats à la liste
        results.append({
            'Animal': trial_row['animal'],
            'Rec': trial_row['rec'],
            'Trial': trial_row['trial'],
            'Period': period,
            'Max_Znorm': Max_Znorm,
            'Associated Temperature': associated_temp,
            'Stim Time': stim_time,
            'Age': trial_row['age'],
            'Sexe': trial_row['sexe'],
            'Weight': trial_row['weight']
        })

# Convertir la liste de résultats en DataFrame
df_max_values = pd.DataFrame(results)

# Définir le chemin où sauvegarder le fichier Excel
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Max_Znorm_by_Period_cool.xlsx"

# Sauvegarder le DataFrame dans un fichier Excel
df_max_values.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les données ont été sauvegardées dans {output_file_path}")


### Supression des Periods index for stat on periods

In [ ]:
import pandas as pd

# Charger le fichier Excel
file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Max_Znorm_by_Period_cool.xlsx"
df = pd.read_excel(file_path)

# Vérifier les colonnes disponibles
print(df.columns)

# Extraire les valeurs de 'Period' sans les suffixes '_index'
df['Period'] = df['Period'].str.extract(r'([A-Z]+)')[0]

# En utilisant str.replace pour supprimer l'indice dans 'Period'
df['Period'] = df['Period'].str.replace(r'_(\d+)', '', regex=True)

# Renommer le fichier de sortie
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Max_Znorm_by_Period_wo_index_cool.xlsx"
df.to_excel(output_file_path, index=False)

print(f"Le fichier a été enregistré sous : {output_file_path}")


### Moyennage par animaux

In [ ]:
import pandas as pd

# Charger le DataFrame contenant les valeurs d'AUC
input_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Max_Znorm_by_Period_wo_index_cool.xlsx"
df_auc_values = pd.read_excel(input_file_path)

# Calculer la moyenne des AUC par période, animal, sexe et âge
mean_auc_by_animal = df_auc_values.groupby(['Animal', 'Sexe', 'Age', 'Period'], as_index=False)['Max_Znorm'].mean()

# Définir le chemin où sauvegarder le fichier Excel des moyennes
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Max_Znorm_by_Period_animal_cool.xlsx"

# Sauvegarder le DataFrame des moyennes dans un fichier Excel
mean_auc_by_animal.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les moyennes d'AUC par période, sexe et âge ont été sauvegardées dans {output_file_path}")


### Récupération des Data par animal pour period TS

In [ ]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Max_Znorm_by_Period_animal_cool.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Quantif_by_Periods/Max_Znorm_TS_only_animal_cool.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où la colonne 'period' contient 'TS' (partout dans la cellule)
df_filtré = df[df['Period'].str.startswith("TS", na=False)]

# Vérifier le nombre de lignes filtrées
print(f"Nombre de lignes filtrées : {len(df_filtré)}")

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
if not df_filtré.empty:
    df_filtré.to_excel(output_file_path, index=False)
    print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")
else:
    print("Aucune ligne correspondante pour 'TS' trouvée.")

## 5.2. For hot stim

In [ ]:
import pandas as pd
import numpy as np
from scipy.integrate import trapezoid

# Charger le dataset depuis le fichier Excel
file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Df_hot_Zscore_classic_hot_stim.xlsx"
df = pd.read_excel(file_path)


# On inclut aussi les colonnes 'age', 'sexe', et 'weight' dans les données uniques
trials_unique = df[['animal', 'rec', 'trial', 'age', 'sexe', 'weight']].drop_duplicates().reset_index(drop=True)

# Créer une liste pour stocker les résultats
results = []

# Itérer sur chaque trial unique
for _, trial_row in trials_unique.iterrows():
    # Filtrer le DataFrame pour obtenir les données de ce trial unique
    trial_data = df[(df['animal'] == trial_row['animal']) & 
                    (df['rec'] == trial_row['rec']) & 
                    (df['trial'] == trial_row['trial'])]
    
    # Grouper les données filtrées par période
    grouped = trial_data.groupby('period')
    
    # Itérer sur chaque période ('TS', 'TB', 'PTS')
    for period, group in grouped:
        # Trouver la ligne avec la valeur maximale de Z-score_norm
        max_row = group.loc[group['Z-score_norm'].idxmax()]
        
        # Extraire les informations nécessaires
        Max_Znorm = max_row['Z-score_norm']
        associated_temp = max_row['temperature']
        stim_time = max_row['Stim_Time']
        
        # Ajouter les résultats à la liste
        results.append({
            'Animal': trial_row['animal'],
            'Rec': trial_row['rec'],
            'Trial': trial_row['trial'],
            'Period': period,
            'Max_Znorm': Max_Znorm,
            'Associated Temperature': associated_temp,
            'Stim Time': stim_time,
            'Age': trial_row['age'],
            'Sexe': trial_row['sexe'],
            'Weight': trial_row['weight']
        })

# Convertir la liste de résultats en DataFrame
df_max_values = pd.DataFrame(results)

# Définir le chemin où sauvegarder le fichier Excel
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Max_Znorm_by_Period_hot.xlsx"

# Sauvegarder le DataFrame dans un fichier Excel
df_max_values.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les données ont été sauvegardées dans {output_file_path}")


### Supression des Periods index for stat on periods

In [ ]:
import pandas as pd

# Charger le fichier Excel
file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Max_Znorm_by_Period_hot.xlsx"
df = pd.read_excel(file_path)

# Vérifier les colonnes disponibles
print(df.columns)

# Supposons que la colonne contenant les périodes s'appelle 'Period'
# Extraire les valeurs de 'Period' sans les suffixes '_index'
df['Period'] = df['Period'].str.extract(r'([A-Z]+)')[0]

# # Créer une nouvelle colonne 'Period_index' avec les indices
# df['Period_index'] = df['Period'].str.extract(r'_(\d+)')[0]

# Supprimer l'indice de la colonne 'Period'
# En utilisant str.replace pour supprimer l'indice dans 'Period'
df['Period'] = df['Period'].str.replace(r'_(\d+)', '', regex=True)

# Renommer le fichier de sortie
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Max_Znorm_by_Period_wo_index_hot.xlsx"
df.to_excel(output_file_path, index=False)

print(f"Le fichier a été enregistré sous : {output_file_path}")


### Moyennage par animaux

In [ ]:
import pandas as pd

# Charger le DataFrame contenant les valeurs d'AUC
input_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Max_Znorm_by_Period_wo_index_hot.xlsx"
df_auc_values = pd.read_excel(input_file_path)

# Calculer la moyenne des AUC par période, animal, sexe et âge
mean_auc_by_animal = df_auc_values.groupby(['Animal', 'Sexe', 'Age', 'Period'], as_index=False)['Max_Znorm'].mean()

# Définir le chemin où sauvegarder le fichier Excel des moyennes
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Max_Znorm_by_Period_animal_hot.xlsx"

# Sauvegarder le DataFrame des moyennes dans un fichier Excel
mean_auc_by_animal.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les moyennes d'AUC par période, sexe et âge ont été sauvegardées dans {output_file_path}")


### Récupération des Data par animal pour period TS

In [ ]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Max_Znorm_by_Period_animal_hot.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Quantif_by_Periods/Max_Znorm_TS_only_animal_hot.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où la colonne 'period' contient 'TS' (partout dans la cellule)
df_filtré = df[df['Period'].str.startswith("TS", na=False)]

# Vérifier le nombre de lignes filtrées
print(f"Nombre de lignes filtrées : {len(df_filtré)}")

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
if not df_filtré.empty:
    df_filtré.to_excel(output_file_path, index=False)
    print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")
else:
    print("Aucune ligne correspondante pour 'TS' trouvée.")

#

#

# 6. Df construction for the Mean Znorm

## 6.1. For cool stim

In [ ]:
import pandas as pd

# Charger le dataset depuis le fichier Excel
file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Df_thermic_Zscore_classic_cool_stim.xlsx"
df = pd.read_excel(file_path)

# Assurez-vous que la colonne 'Z-score_norm' est de type numérique
df['Z-score_norm'] = pd.to_numeric(df['Z-score_norm'], errors='coerce')

# Extraire les essais uniques (en fonction de 'animal', 'rec', 'trial')
trials_unique = df[['animal', 'rec', 'trial', 'age', 'sexe', 'weight']].drop_duplicates().reset_index(drop=True)

# Créer une liste pour stocker les résultats
results = []

# Itérer sur chaque essai unique
for _, trial_row in trials_unique.iterrows():
    # Filtrer le DataFrame pour obtenir les données de cet essai unique
    trial_data = df[(df['animal'] == trial_row['animal']) & 
                    (df['rec'] == trial_row['rec']) & 
                    (df['trial'] == trial_row['trial'])]
    
    # Grouper les données filtrées par période
    grouped = trial_data.groupby('period')
    
    # Itérer sur chaque période ('TS', 'TB', 'PTS')
    for period, group in grouped:
        # Calculer la moyenne et l'écart-type de Z-score_norm pour cette période
        mean_zscore = group['Z-score_norm'].mean()
        
        # Ajouter les résultats à la liste
        results.append({
            'Animal': trial_row['animal'],
            'Rec': trial_row['rec'],
            'Trial': trial_row['trial'],
            'Period': period,
            'Mean_Znorm': mean_zscore,
            'Age': trial_row['age'],
            'Sexe': trial_row['sexe'],
            'Weight': trial_row['weight']
        })

# Convertir la liste de résultats en DataFrame
df_mean_sd_values = pd.DataFrame(results)

# Définir le chemin où sauvegarder le fichier Excel
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Mean_Znorm_by_Period_cool.xlsx"

# Sauvegarder le DataFrame dans un fichier Excel
df_mean_sd_values.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les données ont été sauvegardées dans {output_file_path}")


#### Supression des Periods index for stat on periods

In [ ]:
import pandas as pd

# Charger le fichier Excel
file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Mean_Znorm_by_Period_cool.xlsx"
df = pd.read_excel(file_path)

# Vérifier les colonnes disponibles
print(df.columns)

# Extraire les valeurs de 'Period' sans les suffixes '_index'
df['Period'] = df['Period'].str.extract(r'([A-Z]+)')[0]

# # Créer une nouvelle colonne 'Period_index' avec les indices
# df['Period_index'] = df['Period'].str.extract(r'_(\d+)')[0]

# En utilisant str.replace pour supprimer l'indice dans 'Period'
df['Period'] = df['Period'].str.replace(r'_(\d+)', '', regex=True)

# Renommer le fichier de sortie
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Mean_Znorm_by_Period_wo_index_cool.xlsx"
df.to_excel(output_file_path, index=False)

print(f"Le fichier a été enregistré sous : {output_file_path}")


#### Moyennage par animaux

In [ ]:
import pandas as pd

# Charger le DataFrame contenant les valeurs d'AUC
input_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Mean_Znorm_by_Period_wo_index_cool.xlsx"
df_auc_values = pd.read_excel(input_file_path)

# Calculer la moyenne des AUC par période, animal, sexe et âge
mean_auc_by_animal = df_auc_values.groupby(['Animal', 'Sexe', 'Age', 'Period'], as_index=False)['Mean_Znorm'].mean()

# Définir le chemin où sauvegarder le fichier Excel des moyennes
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Mean_Znorm_by_Period_animal_cool.xlsx"

# Sauvegarder le DataFrame des moyennes dans un fichier Excel
mean_auc_by_animal.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les moyennes de mean Znorm par période, sexe et âge ont été sauvegardées dans {output_file_path}")


#### Récupération des Data par animal pour period TS

In [ ]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Mean_Znorm_by_Period_animal_cool.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Quantif_by_Periods/Mean_Znorm_TS_only_animal_cool.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où la colonne 'period' contient 'TS' (partout dans la cellule)
df_filtré = df[df['Period'].str.startswith("TS", na=False)]

# Vérifier le nombre de lignes filtrées
print(f"Nombre de lignes filtrées : {len(df_filtré)}")

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
if not df_filtré.empty:
    df_filtré.to_excel(output_file_path, index=False)
    print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")
else:
    print("Aucune ligne correspondante pour 'TS' trouvée.")

## 6.2. For hot stim

In [ ]:
import pandas as pd

# Charger le dataset depuis le fichier Excel
file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Df_hot_Zscore_classic_hot_stim.xlsx"
df = pd.read_excel(file_path)

# Assurez-vous que la colonne 'Z-score_norm' est de type numérique
df['Z-score_norm'] = pd.to_numeric(df['Z-score_norm'], errors='coerce')

# Extraire les essais uniques (en fonction de 'animal', 'rec', 'trial')
trials_unique = df[['animal', 'rec', 'trial', 'age', 'sexe', 'weight']].drop_duplicates().reset_index(drop=True)

# Créer une liste pour stocker les résultats
results = []

# Itérer sur chaque essai unique
for _, trial_row in trials_unique.iterrows():
    # Filtrer le DataFrame pour obtenir les données de cet essai unique
    trial_data = df[(df['animal'] == trial_row['animal']) & 
                    (df['rec'] == trial_row['rec']) & 
                    (df['trial'] == trial_row['trial'])]
    
    # Grouper les données filtrées par période
    grouped = trial_data.groupby('period')
    
    # Itérer sur chaque période ('TS', 'TB', 'PTS')
    for period, group in grouped:
        # Calculer la moyenne et l'écart-type de Z-score_norm pour cette période
        mean_zscore = group['Z-score_norm'].mean()
        
        # Ajouter les résultats à la liste
        results.append({
            'Animal': trial_row['animal'],
            'Rec': trial_row['rec'],
            'Trial': trial_row['trial'],
            'Period': period,
            'Mean_Znorm': mean_zscore,
            'Age': trial_row['age'],
            'Sexe': trial_row['sexe'],
            'Weight': trial_row['weight']
        })

# Convertir la liste de résultats en DataFrame
df_mean_sd_values = pd.DataFrame(results)

# Définir le chemin où sauvegarder le fichier Excel
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Mean_Znorm_by_Period_hot.xlsx"

# Sauvegarder le DataFrame dans un fichier Excel
df_mean_sd_values.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les données ont été sauvegardées dans {output_file_path}")


### Supression des Periods index for stat on periods

In [ ]:
import pandas as pd

# Charger le fichier Excel
file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Mean_Znorm_by_Period_hot.xlsx"
df = pd.read_excel(file_path)

# Vérifier les colonnes disponibles
print(df.columns)

# Supposons que la colonne contenant les périodes s'appelle 'Period'
# Extraire les valeurs de 'Period' sans les suffixes '_index'
df['Period'] = df['Period'].str.extract(r'([A-Z]+)')[0]

# # Créer une nouvelle colonne 'Period_index' avec les indices
# df['Period_index'] = df['Period'].str.extract(r'_(\d+)')[0]

# Supprimer l'indice de la colonne 'Period'
# En utilisant str.replace pour supprimer l'indice dans 'Period'
df['Period'] = df['Period'].str.replace(r'_(\d+)', '', regex=True)

# Renommer le fichier de sortie
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Mean_Znorm_by_Period_wo_index_hot.xlsx"
df.to_excel(output_file_path, index=False)

print(f"Le fichier a été enregistré sous : {output_file_path}")


### Moyennage par animaux

In [ ]:
import pandas as pd

# Charger le DataFrame contenant les valeurs d'AUC
input_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Mean_Znorm_by_Period_wo_index_hot.xlsx"
df_auc_values = pd.read_excel(input_file_path)

# Calculer la moyenne des AUC par période, animal, sexe et âge
mean_auc_by_animal = df_auc_values.groupby(['Animal', 'Sexe', 'Age', 'Period'], as_index=False)['Mean_Znorm'].mean()

# Définir le chemin où sauvegarder le fichier Excel des moyennes
output_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Mean_Znorm_by_Period_animal_hot.xlsx"

# Sauvegarder le DataFrame des moyennes dans un fichier Excel
mean_auc_by_animal.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les moyennes de mean Znorm par période, sexe et âge ont été sauvegardées dans {output_file_path}")


### Récupération des Data par animal pour period TS

In [ ]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = r"G:\PhD\Experimentation\Calcium_imaging_WF\analysis_DFF\raw_data_cool_vs_hot\Quantif_by_Periods\Mean_Znorm_by_Period_animal_hot.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Quantif_by_Periods/Mean_Znorm_TS_only_animal_hot.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où la colonne 'period' contient 'TS' (partout dans la cellule)
df_filtré = df[df['Period'].str.startswith("TS", na=False)]

# Vérifier le nombre de lignes filtrées
print(f"Nombre de lignes filtrées : {len(df_filtré)}")

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
if not df_filtré.empty:
    df_filtré.to_excel(output_file_path, index=False)
    print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")
else:
    print("Aucune ligne correspondante pour 'TS' trouvée.")

#

#

# 7. Df for Responsive rate 

## 7.1. Creation of the directory for saving

In [4]:
import os

# Définir le chemin de base et le répertoire à créer
base_dir = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot"
save_dir = os.path.join(base_dir, "Thresholding_cool_vs_hot")

# Vérifier si le répertoire existe, sinon le créer
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"Le répertoire '{save_dir}' a été créé avec succès.")
else:
    print(f"Le répertoire '{save_dir}' existe déjà.")

Le répertoire 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot\Thresholding_cool_vs_hot' existe déjà.


## 7.2. Recuperation of Threshold value

In [3]:
import pandas as pd


# Chemin vers le fichier des seuils (mean_max_z_score)
Threshold_file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_shuffling_threshold/global_distribution/Threshold_value.xlsx'

# Lire les données des seuils pour chaque animal et rec
df_threshold = pd.read_excel(Threshold_file_path)

# Obtenir la valmeur de Threshold
Threshold = df_threshold['Value'].values

print (Threshold)

[1.85966108]


## 7.3. For cool stim

#### New DF witout TS for classic Z

In [ ]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_thermic_Zscore_classic_cool.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_thermic_Zscore_wo_TS_cool.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où 'period' ne commence pas par "TS_"
df_filtered = df[~df['period'].str.startswith("TS_", na=False)]

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
df_filtered.to_excel(output_file_path, index=False)

print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")


#### Binarisation of response rate with Znorm and unique threshold

In [8]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_thermic_Zscore_classic_cool_stim.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
response_rate_results = []

# Définir le seuil fixe
threshold = Threshold

# Liste des animaux
animals = df['animal'].unique()

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Filtrer les lignes qui commencent par "TS_"
            df_ts = df_trial[df_trial['period'].str.startswith("TS_")]
            
            # Vérifier s'il y a des données dans la période "TS"
            if not df_ts.empty:
                # Lire les valeurs de Z-score dans la période "TS"
                z_scores_ts = df_ts['Z-score_norm'].values
                
                # Vérifier si au moins une valeur de Z-score dépasse le seuil
                response_rate = 1 if (z_scores_ts > threshold).any() else 0
                
                # Ajouter les résultats à la liste
                response_rate_results.append({
                    'animal': animal,
                    'rec': rec,
                    'trial': trial,
                    'age': age,    # Ajout de la colonne 'age'
                    'sexe': sexe,  # Ajout de la colonne 'sexe'
                    'response_rate': response_rate
                })

# Convertir les résultats en DataFrame
response_rate_df = pd.DataFrame(response_rate_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Thresholding_cool_vs_hot/Thq_responsive_rate_cool.xlsx'
response_rate_df.to_excel(output_file_path, index=False)

# Afficher les résultats
print(response_rate_df)


        animal        rec    trial age sexe  response_rate
0   2023.11.08  Thermic_3  Trial_1  P6    M              0
1   2023.11.08  Thermic_3  Trial_2  P6    M              0
2   2023.11.08  Thermic_3  Trial_3  P6    M              0
3   2023.11.08  Thermic_3  Trial_4  P6    M              0
4   2023.11.08  Thermic_3  Trial_5  P6    M              1
..         ...        ...      ...  ..  ...            ...
67  2023.11.10  Thermic_5  Trial_4  P8    F              0
68  2023.11.10  Thermic_5  Trial_5  P8    F              1
69  2023.11.10  Thermic_5  Trial_6  P8    F              0
70  2023.11.10  Thermic_5  Trial_7  P8    F              1
71  2023.11.10  Thermic_5  Trial_8  P8    F              1

[72 rows x 6 columns]


#### Coding the Responsive Rate for each period and trial

In [9]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_thermic_Zscore_classic_cool_stim.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
response_rate_results = []

# Définir le seuil fixe
threshold = Threshold

# Liste des animaux
animals = df['animal'].unique()

# Liste des périodes à analyser
periods = ['TB', 'TS', 'PTS']

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Parcourir les périodes (TB, TS, PTS)
            for period in periods:
                # Filtrer les données pour la période actuelle
                df_period = df_trial[df_trial['period'].str.startswith(period)]
                
                # Vérifier s'il y a des données dans cette période
                if not df_period.empty:
                    # Lire les valeurs de Z-score dans la période
                    z_scores_period = df_period['Z-score_norm'].values
                    
                    # Vérifier si au moins une valeur de Z-score dépasse le seuil
                    response_rate = 1 if (z_scores_period > threshold).any() else 0
                    
                    # Ajouter les résultats à la liste
                    response_rate_results.append({
                        'animal': animal,
                        'rec': rec,
                        'trial': trial,
                        'period': period,  # Ajout de la période
                        'age': age,    # Ajout de la colonne 'age'
                        'sexe': sexe,  # Ajout de la colonne 'sexe'
                        'response_rate': response_rate
                    })

# Convertir les résultats en DataFrame
response_rate_df = pd.DataFrame(response_rate_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Thresholding_cool_vs_hot/Thq_responsive_rate_by_period_cool.xlsx'
response_rate_df.to_excel(output_file_path, index=False)

# Afficher les résultats
print(response_rate_df)


         animal        rec    trial period age sexe  response_rate
0    2023.11.08  Thermic_3  Trial_1     TB  P6    M              1
1    2023.11.08  Thermic_3  Trial_1     TS  P6    M              0
2    2023.11.08  Thermic_3  Trial_1    PTS  P6    M              1
3    2023.11.08  Thermic_3  Trial_2     TB  P6    M              0
4    2023.11.08  Thermic_3  Trial_2     TS  P6    M              0
..          ...        ...      ...    ...  ..  ...            ...
211  2023.11.10  Thermic_5  Trial_7     TS  P8    F              1
212  2023.11.10  Thermic_5  Trial_7    PTS  P8    F              1
213  2023.11.10  Thermic_5  Trial_8     TB  P8    F              0
214  2023.11.10  Thermic_5  Trial_8     TS  P8    F              1
215  2023.11.10  Thermic_5  Trial_8    PTS  P8    F              1

[216 rows x 7 columns]


#### Coding the Normalized response rate

In [10]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_thermic_Zscore_classic_cool_stim.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
norm_response_rate_results = []

# Liste des animaux
animals = df['animal'].unique()

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Parcourir chaque période du trial
            for period in df_trial['period'].unique():
                # Filtrer les données pour la période actuelle
                df_period = df_trial[df_trial['period'] == period]
                
                # Lire les valeurs de Z-score pour la période
                z_scores_period = df_period['Z-score_norm'].values
                
                # Définir un seuil fixe de 1.91 (au lieu de mean_95_percent)
                threshold = Threshold
                
                # Calculer le nombre de Z-scores supérieurs au seuil
                numb_of_sup_Z = (z_scores_period > threshold).sum()
                
                # Calculer le nombre total de Z-scores dans la période
                numb_of_tot_Z = len(z_scores_period)
                
                # Calculer le taux de réponse normalisé
                Norm_response_rate = (numb_of_sup_Z / numb_of_tot_Z) * 100 if numb_of_tot_Z > 0 else 0
                
                # Ajouter les résultats à la liste
                norm_response_rate_results.append({
                    'animal': animal,
                    'rec': rec,
                    'trial': trial,
                    'period': period,
                    'age': age,    # Ajout de la colonne 'age'
                    'sexe': sexe,  # Ajout de la colonne 'sexe'
                    'numb_of_sup_Z': numb_of_sup_Z,
                    'numb_of_tot_Z': numb_of_tot_Z,
                    'Norm_response_rate': Norm_response_rate
                })

# Convertir les résultats en DataFrame
norm_response_rate_df = pd.DataFrame(norm_response_rate_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Thresholding_cool_vs_hot/Thq_Norm_response_rate_cool.xlsx'
norm_response_rate_df.to_excel(output_file_path, index=False)

print(f"Les résultats ont été sauvegardés dans {output_file_path}")

# # Afficher les résultats
# print(norm_response_rate_df)


Les résultats ont été sauvegardés dans D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Thresholding_cool_vs_hot/Thq_Norm_response_rate_cool.xlsx


#### Coding the Magnitude of the response

In [ ]:
#### Coding the Magnitude of the response
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_thermic_Zscore_classic_cool_stim.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
response_magnitude_results = []

# Liste des animaux
animals = df['animal'].unique()

# Définir le seuil pour les Z-scores
threshold = Threshold

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Parcourir chaque période du trial
            for period in df_trial['period'].unique():
                # Filtrer les données pour la période actuelle
                df_period = df_trial[df_trial['period'] == period]
                
                # Lire les valeurs de Z-score pour la période
                z_scores_period = df_period['Z-score_norm'].values
                
                # Filtrer les Z-scores au-dessus du seuil
                z_scores_above_threshold = z_scores_period[z_scores_period > threshold]
                
                # Calculer la magnitude de la réponse normalisée
                # Diviser la somme des Z-scores au-dessus du seuil par le nombre total de valeurs dans la période
                if len(z_scores_period) > 0:  # Éviter la division par zéro
                    response_magnitude = z_scores_above_threshold.sum() / len(z_scores_period)
                else:
                    response_magnitude = 0  # Si la période est vide, définir à 0
                
                # Ajouter les résultats à la liste
                response_magnitude_results.append({
                    'animal': animal,
                    'rec': rec,
                    'trial': trial,
                    'period': period,
                    'age': age,    # Ajout de la colonne 'age'
                    'sexe': sexe,  # Ajout de la colonne 'sexe'
                    'response_magnitude': response_magnitude
                })

# Convertir les résultats en DataFrame
response_magnitude_df = pd.DataFrame(response_magnitude_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Thresholding_cool_vs_hot/Thq_Response_Magnitude_cool.xlsx'
response_magnitude_df.to_excel(output_file_path, index=False)

print(f"Les résultats ont été sauvegardés dans {output_file_path}")

# Afficher un aperçu des résultats
print(response_magnitude_df)


Les résultats ont été sauvegardés dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Thresholding_cool_vs_hot/Cool_Response_Magnitude.xlsx
         animal        rec    trial period age sexe  response_magnitude
0    2023.11.08  Thermic_3  Trial_1   TB_1  P6    M            0.111895
1    2023.11.08  Thermic_3  Trial_1   TS_1  P6    M            0.000000
2    2023.11.08  Thermic_3  Trial_1  PTS_1  P6    M            1.379763
3    2023.11.08  Thermic_3  Trial_2   TB_2  P6    M            0.000000
4    2023.11.08  Thermic_3  Trial_2   TS_2  P6    M            0.000000
..          ...        ...      ...    ...  ..  ...                 ...
211  2023.11.10  Thermic_5  Trial_7   TS_7  P8    F            0.191103
212  2023.11.10  Thermic_5  Trial_7  PTS_7  P8    F            0.569841
213  2023.11.10  Thermic_5  Trial_8   TB_8  P8    F            0.000000
214  2023.11.10  Thermic_5  Trial_8   TS_8  P8    F            0.764637
215  2023.11.10  Thermic_5  Trial_8  PT

## 7.4. For hot stimulation

#### New DF witout TS for classic Z

In [ ]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_hot_Zscore_classic_hot.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_hot_Zscore_wo_TS_hot.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où 'period' ne commence pas par "TS_"
df_filtered = df[~df['period'].str.startswith("TS_", na=False)]

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
df_filtered.to_excel(output_file_path, index=False)

print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")


#### Coding of Responsive Rate for each unique trials over TS period

In [11]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_hot_Zscore_classic_hot_stim.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
response_rate_results = []

# Liste des animaux
animals = df['animal'].unique()

# Définition du seuil fixe
threshold = Threshold

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Filtrer les lignes qui commencent par "TS_"
            df_ts = df_trial[df_trial['period'].str.startswith("TS_")]
            
            # Vérifier s'il y a des données dans la période "TS"
            if not df_ts.empty:
                # Lire les valeurs de Z-score dans la période "TS"
                z_scores_ts = df_ts['Z-score_norm'].values
                
                # Vérifier si au moins une valeur de Z-score dépasse le seuil fixe
                response_rate = 1 if (z_scores_ts > threshold).any() else 0
                
                # Ajouter les résultats à la liste
                response_rate_results.append({
                    'animal': animal,
                    'rec': rec,
                    'trial': trial,
                    'age': age,    # Ajout de la colonne 'age'
                    'sexe': sexe,  # Ajout de la colonne 'sexe'
                    'response_rate': response_rate
                })

# Convertir les résultats en DataFrame
response_rate_df = pd.DataFrame(response_rate_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Thresholding_cool_vs_hot/Thq_responsive_rate_hot.xlsx'
response_rate_df.to_excel(output_file_path, index=False)

# Afficher les résultats
print(response_rate_df)


        animal    rec    trial age sexe  response_rate
0   2023.11.08  hot_1  Trial_1  P6    M              0
1   2023.11.08  hot_1  Trial_2  P6    M              0
2   2023.11.08  hot_1  Trial_3  P6    M              0
3   2023.11.08  hot_1  Trial_4  P6    M              0
4   2023.11.08  hot_1  Trial_5  P6    M              0
..         ...    ...      ...  ..  ...            ...
67  2023.11.10  hot_3  Trial_4  P8    F              0
68  2023.11.10  hot_3  Trial_5  P8    F              1
69  2023.11.10  hot_3  Trial_6  P8    F              0
70  2023.11.10  hot_3  Trial_7  P8    F              0
71  2023.11.10  hot_3  Trial_8  P8    F              0

[72 rows x 6 columns]


#### Coding the Responsive Rate for each period and trial

In [12]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_hot_Zscore_classic_hot_stim.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
response_rate_results = []

# Liste des animaux
animals = df['animal'].unique()

# Liste des périodes à analyser
periods = ['TB', 'TS', 'PTS']

# Seuil fixe
threshold = Threshold

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Parcourir les périodes (TB, TS, PTS)
            for period in periods:
                # Filtrer les données pour la période actuelle
                df_period = df_trial[df_trial['period'].str.startswith(period)]
                
                # Vérifier s'il y a des données dans cette période
                if not df_period.empty:
                    # Lire les valeurs de Z-score dans la période
                    z_scores_period = df_period['Z-score_norm'].values
                    
                    # Vérifier si au moins une valeur de Z-score dépasse le seuil
                    response_rate = 1 if (z_scores_period > threshold).any() else 0
                    
                    # Ajouter les résultats à la liste
                    response_rate_results.append({
                        'animal': animal,
                        'rec': rec,
                        'trial': trial,
                        'period': period,  # Ajout de la période
                        'age': age,    # Ajout de la colonne 'age'
                        'sexe': sexe,  # Ajout de la colonne 'sexe'
                        'response_rate': response_rate
                    })

# Convertir les résultats en DataFrame
response_rate_df = pd.DataFrame(response_rate_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Thresholding_cool_vs_hot/Thq_responsive_rate_by_period_hot.xlsx'
response_rate_df.to_excel(output_file_path, index=False)

# Afficher les résultats
print(response_rate_df)


         animal    rec    trial period age sexe  response_rate
0    2023.11.08  hot_1  Trial_1     TB  P6    M              1
1    2023.11.08  hot_1  Trial_1     TS  P6    M              0
2    2023.11.08  hot_1  Trial_1    PTS  P6    M              1
3    2023.11.08  hot_1  Trial_2     TB  P6    M              1
4    2023.11.08  hot_1  Trial_2     TS  P6    M              0
..          ...    ...      ...    ...  ..  ...            ...
211  2023.11.10  hot_3  Trial_7     TS  P8    F              0
212  2023.11.10  hot_3  Trial_7    PTS  P8    F              0
213  2023.11.10  hot_3  Trial_8     TB  P8    F              0
214  2023.11.10  hot_3  Trial_8     TS  P8    F              0
215  2023.11.10  hot_3  Trial_8    PTS  P8    F              0

[216 rows x 7 columns]


#### Coding the Normalized response rate

In [13]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_hot_Zscore_classic_hot_stim.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
norm_response_rate_results = []

# Liste des animaux
animals = df['animal'].unique()

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Parcourir chaque période du trial
            for period in df_trial['period'].unique():
                # Filtrer les données pour la période actuelle
                df_period = df_trial[df_trial['period'] == period]
                
                # Lire les valeurs de Z-score pour la période
                z_scores_period = df_period['Z-score_norm'].values
                
                # Définir le seuil à 1.91
                threshold = Threshold
                
                # Calculer le nombre de Z-scores supérieurs au seuil
                numb_of_sup_Z = (z_scores_period > threshold).sum()
                
                # Calculer le nombre total de Z-scores dans la période
                numb_of_tot_Z = len(z_scores_period)
                
                # Calculer le taux de réponse normalisé
                Norm_response_rate = (numb_of_sup_Z / numb_of_tot_Z) * 100 if numb_of_tot_Z > 0 else 0
                
                # Ajouter les résultats à la liste
                norm_response_rate_results.append({
                    'animal': animal,
                    'rec': rec,
                    'trial': trial,
                    'period': period,
                    'age': age,    # Ajout de la colonne 'age'
                    'sexe': sexe,  # Ajout de la colonne 'sexe'
                    'numb_of_sup_Z': numb_of_sup_Z,
                    'numb_of_tot_Z': numb_of_tot_Z,
                    'Norm_response_rate': Norm_response_rate
                })

# Convertir les résultats en DataFrame
norm_response_rate_df = pd.DataFrame(norm_response_rate_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Thresholding_cool_vs_hot/Thq_Norm_response_rate_hot.xlsx'
norm_response_rate_df.to_excel(output_file_path, index=False)

print(f"Les résultats ont été sauvegardés dans {output_file_path}")

# # Afficher les résultats
# print(norm_response_rate_df)


Les résultats ont été sauvegardés dans D:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Thresholding_cool_vs_hot/Thq_Norm_response_rate_hot.xlsx


#### Coding the Magnitude of the response

In [7]:
#### Coding the Magnitude of the response
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Df_hot_Zscore_classic_hot.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
response_magnitude_results = []

# Liste des animaux
animals = df['animal'].unique()

# Définir le seuil pour les Z-scores
threshold = Threshold

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Parcourir chaque période du trial
            for period in df_trial['period'].unique():
                # Filtrer les données pour la période actuelle
                df_period = df_trial[df_trial['period'] == period]
                
                # Lire les valeurs de Z-score pour la période
                z_scores_period = df_period['Z-score_norm'].values
                
                # Filtrer les Z-scores au-dessus du seuil
                z_scores_above_threshold = z_scores_period[z_scores_period > threshold]
                
                # Calculer la magnitude de la réponse normalisée
                # Diviser la somme des Z-scores au-dessus du seuil par le nombre total de valeurs dans la période
                if len(z_scores_period) > 0:  # Éviter la division par zéro
                    response_magnitude = z_scores_above_threshold.sum() / len(z_scores_period)
                else:
                    response_magnitude = 0  # Si la période est vide, définir à 0
                
                # Ajouter les résultats à la liste
                response_magnitude_results.append({
                    'animal': animal,
                    'rec': rec,
                    'trial': trial,
                    'period': period,
                    'age': age,    # Ajout de la colonne 'age'
                    'sexe': sexe,  # Ajout de la colonne 'sexe'
                    'response_magnitude': response_magnitude
                })

# Convertir les résultats en DataFrame
response_magnitude_df = pd.DataFrame(response_magnitude_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Thresholding_cool_vs_hot/Thq_Response_Magnitude_hot.xlsx'
response_magnitude_df.to_excel(output_file_path, index=False)

print(f"Les résultats ont été sauvegardés dans {output_file_path}")

# Afficher un aperçu des résultats
print(response_magnitude_df)


Les résultats ont été sauvegardés dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_cool_vs_hot/Thresholding_cool_vs_hot/hot_Response_Magnitude.xlsx
         animal    rec    trial  period age sexe  response_magnitude
0    2023.11.08  hot_1  Trial_1  ITST_1  P6    M            0.000000
1    2023.11.08  hot_1  Trial_1    TB_1  P6    M            0.076769
2    2023.11.08  hot_1  Trial_1    TS_1  P6    M            0.000000
3    2023.11.08  hot_1  Trial_1   PTS_1  P6    M            1.495940
4    2023.11.08  hot_1  Trial_1  ITST_2  P6    M            0.000000
..          ...    ...      ...     ...  ..  ...                 ...
292  2023.11.10  hot_3  Trial_7  ITST_8  P8    F            0.000000
293  2023.11.10  hot_3  Trial_8    TB_8  P8    F            0.000000
294  2023.11.10  hot_3  Trial_8    TS_8  P8    F            0.000000
295  2023.11.10  hot_3  Trial_8   PTS_8  P8    F            0.000000
296  2023.11.10  hot_3  Trial_8  ITST_9  P8    F            0.000000

[29